In [1]:
import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt

#### Load Synthetic Tracks

In [2]:
tt_df = pd.read_csv('/lustre/geocean/WORK/users/alonsoap/personal/TC_Track_Clustering/02_VAR_example/outputs/tracks_params/historical_tracks_params.csv',index_col=0)
tt_df_synth = pd.read_csv('/lustre/geocean/WORK/users/alonsoap/personal/TC_Track_Clustering/02_VAR_example/outputs/tracks_params/synthetic_tracks_params.csv',index_col=0)

syn_tracks = xr.open_dataset('/lustre/geocean/WORK/users/alonsoap/personal/TC_Track_Gen_v3/outputs/daily_wind/NA/tracks_all_NA_2.nc')
syn_tracks = syn_tracks.assign_coords(n_trk=np.arange(syn_tracks.sizes["n_trk"]))
syn_tracks = syn_tracks.rename({"n_trk": "storm"})


#### Emulation

In [3]:
models = ['ec_earth3_veg_lr'] 
scenarios = ['historical', 'ssp245', 'ssp585']

In [4]:
for model in models:
    for ssp in scenarios:
        simulated_daily_bmus = xr.open_dataset(f'outputs/tc_logit/{model}/{ssp}/simulated_tracktypes_{model}_{ssp}.nc')
        simulated_daily_bmus.evbmus_sims.values = simulated_daily_bmus.evbmus_sims.values - 2

        TC_id_assig = np.zeros(simulated_daily_bmus.evbmus_sims.values.shape) * np.nan

        n_days, n_sims = simulated_daily_bmus.evbmus_sims.values.shape
        n_tt = 20

        list_idx_synth_per_tt = []
        for i in range(n_tt):
            aux = tt_df_synth[tt_df_synth['Track_Type'] == i].storm_id.values
            list_idx_synth_per_tt.append(aux)

        list_days = []
        list_sims = []

        
        for j in range(n_sims):
            for i in range(n_days):
                bmus = simulated_daily_bmus.evbmus_sims.values[i,j]
                if bmus == -1:
                    continue
                else:
                    synth_idx_candidates = list_idx_synth_per_tt[bmus]
                    chosen_synth_idx = np.random.choice(synth_idx_candidates)
                    TC_id_assig[i,j] = chosen_synth_idx

                    list_days.append(simulated_daily_bmus.time.values[i])
                    list_sims.append(simulated_daily_bmus.n_sim.values[j])

        clean_vals = TC_id_assig.flatten()
        clean_vals = clean_vals[~np.isnan(clean_vals)]

        emu_tracks_list = []

        for tc_id in clean_vals:
            aux = syn_tracks.sel(storm=int(tc_id))
            emu_tracks_list.append(aux)

        emu_tracks = xr.concat(emu_tracks_list,dim='storm_number')
        emu_tracks['original_storm_id'] = ('storm_number', clean_vals.astype(int))
        emu_tracks['time_emulation'] = ('storm_number', np.array(list_days))
        emu_tracks['sim_number'] = ('storm_number', np.array(list_sims))

        emu_tracks.to_netcdf(f'outputs/tc_logit/{model}/{ssp}/emu_tracks_{model}_{ssp}.nc')

        simulated_daily_bmus['tc_id']= (('time', 'n_sims'), TC_id_assig)
        simulated_daily_bmus.to_netcdf(f'outputs/tc_logit/{model}/{ssp}/simulated_tracktypes_with_tc_id_{model}_{ssp}.nc')